In [2]:
# 知识库的搭建
# 文档解析->分块->embedding->向量数据库
# 1 文档解析

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

In [4]:
# PDF解析
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader('./山鹰国际控股股份公司人力资源管理制度.pdf')
docs = loader.load()

In [5]:
print(len(docs))
print(docs[1])

71
page_content='（此页无正文）
山鹰国际控股股份公司
2023 年1 月15 日
山鹰集团' metadata={'producer': '; modified using iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'WPS Writer', 'creationdate': '2023-01-15T11:46:25+11:46', 'source': './山鹰国际控股股份公司人力资源管理制度.pdf', 'file_path': './山鹰国际控股股份公司人力资源管理制度.pdf', 'total_pages': 71, 'format': 'PDF 1.7', 'title': '', 'author': 'llw', 'subject': '', 'keywords': '', 'moddate': '2023-01-15T20:27:13+08:00', 'trapped': '', 'modDate': "D:20230115202713+08'00'", 'creationDate': "D:20230115114625+11'46'", 'page': 1}


In [6]:
data =[]
for doc in docs:
    data += doc.page_content.split('\u3002')
word_num = [len(txt) for txt in data if len(txt) != 0 ]

In [7]:
import numpy as np
print(np.mean(word_num))
print(np.min(word_num))
print(np.max(word_num))
print(np.percentile(word_num, q=95))

54.764604810996566
3
1299
113.89999999999986


In [8]:
r_spliter  = RecursiveCharacterTextSplitter(chunk_size=200,
                                         chunk_overlap=40,
                                         separators=["\n\n",
                                                    "\n",
                                                    ".",
                                                    "\uff0e",
                                                    "\u3002",
                                                    ",",
                                                    "\uff0c",
                                                    "\u3001"
                                                    ])

In [9]:
split_docs = loader.load_and_split(r_spliter)

In [10]:
print(split_docs[123].page_content)

力资源伙伴或直接考核者的上级进行申诉。人力资源伙伴应
组织有关人员进行调查，将结果反馈给被考核者。
三、
考核结果应用
（一）
考核结果应与考核奖金挂钩，具体规定详见
当年度各单位年度绩效考核发文。
（二）
各单位应对中层及以下人员按职能、模块进
行等级分布。有其他规定的，应按股份公司、事业部或各单
位发布的当年度绩效考核方案执行。
（三）考核结果可作为员工年度加薪、晋升、调职的依
据。
（四）


In [11]:
from langchain_core.documents import Document
documents=[]
for chunk in split_docs:
    document = Document(page_content=chunk.page_content,
                       metadata={"source":"test"})
    documents.append(document)


In [12]:
# 连接数据库 embedding
import chromadb
from langchain.vectorstores import Chroma

In [13]:
chroma_client = chromadb.HttpClient(host = "localhost", port=8000)

In [14]:
from langchain.embeddings import HuggingFaceEmbeddings
model_name = "../../model/gte-large-zh"
model_kwargs = {'device': 'cpu'}  # 根据实际情况选择设备，如 'cuda' 用于 GPU
encode_kwargs = {'normalize_embeddings': False}

# 创建本地嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

C:\Users\00036777\AppData\Local\Temp\ipykernel_14772\1057980759.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [15]:
db = Chroma.from_documents(docs, embeddings)

In [16]:
query="普通员工年假有几天？"
related_docs=db.similarity_search(query,k=2)
print(related_docs)

[Document(metadata={'author': 'llw', 'creationDate': "D:20230115114625+11'46'", 'creationdate': '2023-01-15T11:46:25+11:46', 'creator': 'WPS Writer', 'file_path': './山鹰国际控股股份公司人力资源管理制度.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': "D:20230115202713+08'00'", 'moddate': '2023-01-15T20:27:13+08:00', 'page': 30, 'producer': '; modified using iText® 5.5.9 ©2000-2015 iText Group NV (AGPL-version)', 'source': './山鹰国际控股股份公司人力资源管理制度.pdf', 'subject': '', 'title': '', 'total_pages': 71, 'trapped': ''}, page_content='29\n的，则其只享受在职期间所折算的年休假天数，其余应按照\n事假标准予以扣回。\n（五）各单位根据生产经营的具体情况，并结合员工意\n愿，统筹安排员工年休假，年休假在一个年度内可以集中安\n排，也可以分段安排。确因生产、工作特点有必要跨年度安\n排员工年休假的，可以跨一个年度安排并需在次年3 月31\n日前休完，不得提前支取次年年休假。\n（六）员工自愿放弃年休假的，年底前各单位应与员工\n进行书面确认。\n（七）当年应休年休假天数=入职当年的工作天数/365\n天*本人全年应当享受的年休假天数（折算后不足1 天的，\n不享受年休假）。\n（八）各单位在员工申请事假、病假时，应当安排先休\n年休假。\n第十九条员工请假管理\n一、\n各单位员工请假必须发起《请假申请流程》，员\n工请假按《人力资源审批权责表》审批执行。\n二、\n补签卡管理\n（一）员工补签卡：\n员工因下列情况之一，不能打卡产生的缺勤记录，由本\n人于当天或次日申请补签卡：\n1. 需连续在外工作，不能到公司打卡的；\n2. 确因工作需要，上班前需直接外出开会、办事的；\n山

In [17]:
prompt_template="""
你是企业员工助手，熟悉公司考勤和报销标准等规章制度，需要根据提供的上下文信息context来回答员工的提问。\
请直接回答问题，如果上下文信息context没有和问题相关的信息，请直接回答【不知道，请咨询HR】\
问题：{question}
{context}
回答：
"""

In [18]:
import requests
def LLM_call(API_URL, Model_name, prompt):
    # 构建请求数据
    data = {
        "model": model_name,
        "prompt": prompt
    }
    try:
        # 发送 POST 请求到 Ollama API
        response = requests.post(API_URL, json=data)
        # 检查请求是否成功
        response.raise_for_status()
    
        # 初始化一个空字符串用于存储最终的响应结果
        full_response = ""
        # 遍历响应文本中的每一行
        for line in response.text.strip().split('\n'):
            # 解析当前行的 JSON 数据
            import json
            line_data = json.loads(line)
            # 检查当前行是否包含 'response' 字段
            if 'response' in line_data:
                # 将 'response' 字段的值添加到最终响应结果中
                full_response += line_data['response']
    
        # 打印最终的响应结果
        return full_response

    except requests.RequestException as e:
    # 若请求过程中出现异常，打印错误信息
        return "error"

In [19]:
def create_prompt(query, k =2):
    related_docs=db.similarity_search(query,k)
    context = "\n".join([f"上下文{i+1}:{doc.page_content} \n"\
                        for i, doc in enumerate(related_docs)])
    print(f"问题：{query}")
    #print(f"制度内容：{context}")
    llm_prompt = prompt_template.replace("{question}",query).replace("{context}", context)
    return llm_prompt, context

In [20]:
query="实习期入职需要提交什么材料？"
prompt = create_prompt(query, 5)
# 定义 Ollama API 的地址
API_URL = "http://localhost:11434/api/generate"
model_name = "qwen2.5:3b" 
res = LLM_call(API_URL, model_name, prompt[0])
print("="*80)
print(res)

问题：实习期入职需要提交什么材料？
实习期入职需提交的材料包括：

1. 身份证复印件一份（同时原件进行现场核验）；
2. 推荐表原件、成绩单原件；
3. 应届毕业生证原件及复印件一份；
4. 个人简历一份；
5. 一寸彩色免冠电子照片；
6. 工资卡复印件一份。


In [21]:
# 利用Ragrs进行评估
# 构件评估数据集合（问题/标准答案/上下文信息/生成的答案）
questions=[
    "组织架构的设计原则是什么？",
    "签订无固定期限合同的条件是什么？"
]
ground_truths = [
    "（一） 精简原则：组织架构设计应遵循组织扁平化原则。\
    （二） 效率原则：组织架构设计应符合高效运作管理原则。\
    （三） 清晰原则：组织架构设计应职责明确与不重叠原",
    "1. 在股份公司内连续工作满十年的;\
    2. 各单位员工在该用人单位连续工作满十年且距法定退休年龄不足十年的;\
    3. 劳动法规定的相关情形外，员工连续订立二次固定期限劳动合同续订劳动合同的。"
]

In [22]:
answers=[]
contexts = []

for query in questions:
    prompt = create_prompt(query, 5)
    contexts.append(prompt[1])
    res = LLM_call(API_URL, model_name, prompt[0])
    answers.append(res)
    

问题：组织架构的设计原则是什么？
问题：签订无固定期限合同的条件是什么？


In [23]:
type(contexts)

list

In [24]:
from datasets import Dataset
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth":ground_truths
}

In [25]:
dataset = Dataset.from_dict(data)

In [26]:
import pandas as pd
df = pd.DataFrame(data)

In [27]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

In [28]:
from ragas import evaluate
from ragas import RunConfig

In [29]:
# 2通过transformer调用模型
from transformers import AutoModel, AutoTokenizer
model_path = '../../model/Qwen2.5-0.5B-Instruct/'
model = AutoModel.from_pretrained(model_path, trust_remote_code=True)
embedding_model = embeddings

In [30]:
config = RunConfig(timeout=1200, log_tenacity=True)

In [33]:
result = evaluate(
    dataset=df,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy
    ]
)
#df = result.to_pandas()

AttributeError: 'Qwen2Model' object has no attribute 'set_run_config'